In [ ]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

# Panama boundary
countries = ee.FeatureCollection(
    "USDOS/LSIB_SIMPLE/2017"
)

panama = countries.filter(
    ee.Filter.eq('country_na', 'Panama')
)

# ESA WorldCover
worldcover = ee.ImageCollection(
    "ESA/WorldCover/v200"
).first()

landcover = worldcover.select('Map')

# Forest class in ESA WorldCover
# 10 = Tree cover
forest = landcover.eq(10)

# WWF Ecoregions
ecoregions = ee.FeatureCollection(
    "RESOLVE/ECOREGIONS/2017"
)

# Panama dry forests
dry_ecoregion = ecoregions.filter(
    ee.Filter.eq(
        'ECO_NAME',
        'Panamanian dry forests'
    )
)

# Rasterize dry forest polygon
dry_mask = ee.Image.constant(0).paint(
    dry_ecoregion,
    1
).selfMask()

# Dry forest
dry_forest = forest.updateMask(
    dry_mask
)

# Wet forest
wet_mask = dry_mask.unmask(0).eq(0)

wet_forest = forest.updateMask(
    wet_mask
)

# Create classification
classified = ee.Image(0)

# Wet forest = 1
classified = classified.where(
    wet_forest,
    1
)

# Dry forest = 2
classified = classified.where(
    dry_forest,
    2
)

# Visualization
Map = geemap.Map()

Map.centerObject(panama, 7)

Map.addLayer(
    classified.clip(panama),
    {
        'min': 0,
        'max': 2,
        'palette': [
            'white',    # non-forest
            '006400',   # wet forest
            'd4a017'    # dry forest
        ]
    },
    'Forest Type'
)

Map.addLayer(
    panama,
    {},
    'Panama Boundary'
)

Map

Map(center=[8.513881071215563, -80.10553238925849], controls=(WidgetControl(options=['position', 'transparent_…